# pymatgen and Materials Project

Exercises from the ITMO course *Artificial Intelligence in a Chemist's Work*: analysing a crystal structure retrieved from
Materials Project, and computing composition descriptors for a list of inorganic materials.
Running the notebook requires a Materials Project API key in the `MP_API_KEY` environment variable.

In [ ]:
# !pip install pymatgen mp-api
import os
import pandas as pd
import pymatgen.core as pmg
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from mp_api.client import MPRester

# Materials Project API key is read from an environment variable
a = MPRester(os.environ["MP_API_KEY"])

## Task 1. Structure of a Materials Project entry (mp-1205364)

In [ ]:
x = a.get_structure_by_material_id("mp-1205364")
x

Structure Summary
Lattice
    abc : 5.411361367000652 5.411361367000652 7.68509595
 angles : 90.0 90.0 90.60100906523411
 volume : 225.02899171175915
      A : 3.80628905 -3.84642633 -0.0
      B : 3.80628905 3.84642633 0.0
      C : 0.0 0.0 7.68509595
    pbc : True True True
PeriodicSite: Ca (3.806, -0.1108, 1.921) [0.5144, 0.4856, 0.25]
PeriodicSite: Ca (3.806, 0.1108, 5.764) [0.4856, 0.5144, 0.75]
PeriodicSite: Ca (3.806, 3.762, 1.921) [0.01092, 0.9891, 0.25]
PeriodicSite: Ca (3.806, -3.762, 5.764) [0.9891, 0.01092, 0.75]
PeriodicSite: Ti (1.903, 1.923, 3.843) [-0.0, 0.5, 0.5]
PeriodicSite: Ti (1.903, 1.923, 0.0) [-0.0, 0.5, 0.0]
PeriodicSite: Ti (1.903, -1.923, 3.843) [0.5, -0.0, 0.5]
PeriodicSite: Ti (1.903, -1.923, 0.0) [0.5, -0.0, 0.0]
PeriodicSite: O (5.366, -1.884, 1.921) [0.9498, 0.4601, 0.25]
PeriodicSite: O (2.246, 1.884, 5.764) [0.05024, 0.5399, 0.75]
PeriodicSite: O (2.246, -1.884, 1.921) [0.5399, 0.05024, 0.25]
PeriodicSite: O (5.366, 1.884, 5.764) [0.4601, 0.9498, 0.75

In [ ]:
x.formula

'Ca4 Ti4 O12'

In [ ]:
finder_x = SpacegroupAnalyzer(x)
xx = finder_x.get_crystal_system()
xxx = finder_x.get_space_group_symbol()
vol = x.volume

display(f"Crystal system: {xx}")
display(f"Space group: {xxx}")
display(f"Unit cell volume: {vol} Å³")

'Crystal system: orthorhombic'

'Space group: Cmcm'

'Unit cell volume: 225.02899171175915 Å³'

## Task 2. Descriptors for a list of materials

For each formula in `material.csv`: mean electronegativity, mean electron affinity and mean ionic radius of the metals.
The structure is taken from Materials Project when the formula is found there; otherwise the descriptors are computed from the formula itself.

In [ ]:
material_df = pd.read_csv('data/material.csv')

In [ ]:
def calculate_average_electron_affinity(composition):
    total_ea = 0.0
    total_amount = 0.0
    for el_symbol, amount in composition.get_el_amt_dict().items():
        element_obj = pmg.Element(el_symbol)
        if element_obj.electron_affinity is not None:
            total_ea += element_obj.electron_affinity * amount
            total_amount += amount
    return total_ea / total_amount if total_amount > 0 else None

def calculate_average_metal_ionic_radius(composition):
    total_ir = 0.0
    total_metal_amount = 0.0
    for el_symbol, amount in composition.get_el_amt_dict().items():
        element_obj = pmg.Element(el_symbol)
        if element_obj.is_metal and element_obj.average_ionic_radius is not None:
            total_ir += element_obj.average_ionic_radius * amount
            total_metal_amount += amount
    return total_ir / total_metal_amount if total_metal_amount > 0 else None

results_from_csv = []

for index, row in material_df.iterrows():
    formula = row['formula']

    current_formula_data = {
        'Compound ID': formula,
        'Formula': None,
        'Average Electronegativity': None,
        'Average Electron Affinity': None,
        'Average Ionic Radius (Metals)': None
    }

    try:
        materials = a.materials.search(formula=formula, fields=["structure"])

        if materials:
            # structure of the first entry found
            structure = materials[0].structure
            if structure:
                composition = structure.composition
                current_formula_data['Formula'] = structure.formula
                current_formula_data['Average Electronegativity'] = composition.average_electroneg
                current_formula_data['Average Electron Affinity'] = calculate_average_electron_affinity(composition)
                current_formula_data['Average Ionic Radius (Metals)'] = calculate_average_metal_ionic_radius(composition)
            else:
                print(f"No structure returned for formula: {formula}")
        else:
            # not in Materials Project: compute descriptors from the formula
            try:
                composition_from_formula = pmg.Composition(formula)
                print(f"No materials found for formula: {formula} in Materials Project. Computing descriptors directly from pymatgen.Composition.")
                current_formula_data['Formula'] = composition_from_formula.reduced_formula
                current_formula_data['Average Electronegativity'] = composition_from_formula.average_electroneg
                current_formula_data['Average Electron Affinity'] = calculate_average_electron_affinity(composition_from_formula)
                current_formula_data['Average Ionic Radius (Metals)'] = calculate_average_metal_ionic_radius(composition_from_formula)
            except Exception as comp_e:
                print(f"No materials found for formula: {formula}, and pymatgen.Composition failed: {comp_e}")
    except Exception as e:
        print(f"Error while processing {formula}: {e}")
    results_from_csv.append(current_formula_data)

df_material_csv_descriptors = pd.DataFrame(results_from_csv)
display(df_material_csv_descriptors)

No materials found for formula: Cu0.89Zn0.11O in Materials Project. Computing descriptors directly from pymatgen.Composition.


No materials found for formula: Fe0.5Ni0.5Co2O4 in Materials Project. Computing descriptors directly from pymatgen.Composition.
No materials found for formula: Au@CeO2 in Materials Project. Computing descriptors directly from pymatgen.Composition.


,Compound ID,Formula,Average Electronegativity,Average Electron Affinity,Average Ionic Radius (Metals)
0,CoFe2O4,Fe4 Co2 O8,2.757143,0.973311,0.824444
1,Fe2(MoO4)3,Fe16 Mo24 O96,3.024706,1.181267,0.806000
2,Cu0.89Zn0.11O,Zn0.11Cu0.89O1,2.656250,1.246380,0.826600
3,GeO2,Ge2 O4,2.963333,1.384967,NaN
4,Fe0.5Ni0.5Co2O4,Fe0.5Co2Ni0.5O4,2.770000,1.117737,0.777639
5,Au@CeO2,CeAuO2,2.635000,1.457749,1.075000
6,CuInS2,In2 Cu2 S4,2.210000,1.443480,0.880000
7,Eu2O2S,Eu2 S1 O2,2.372000,1.046318,1.198500
8,Co3V2O8,V4 Co6 O16,2.801538,1.133153,0.772000
9,LaCrO3,La2 Cr2 O6,2.616000,1.123363,1.056000


In [ ]:
df_material_csv_descriptors.to_csv('data/material_descriptors.csv', index=False)
print('Table saved to data/material_descriptors.csv')

Table saved to data/material_descriptors.csv
